In [2]:
import pandas as pd
import re
from ollama import chat, ChatResponse
from tqdm import tqdm  # 用於顯示進度條

In [7]:
# --- 設定 ---
FILE_PATH = './data/c3_m_test.xlsx'     # 資料集路徑
MODEL_NAME = 'gemma3:12b'  # 模型名稱

In [8]:
# --- 函數：呼叫 Ollama (使用官方庫) ---
def query_ollama(prompt, model=MODEL_NAME):
    try:
        # 使用 ollama.chat 直接呼叫
        response: ChatResponse = chat(
            model=model,
            messages=[
                {'role': 'user', 'content': prompt}
            ],
            # 您可以在此加入 options，例如 temperature
            options={
                'temperature': 0.0  # 讓輸出結果穩定
            }
        )
        # 回傳模型的內容
        return response.message.content
    except Exception as e:
        print(f"Error calling Ollama: {e}")
        return ""

In [9]:
# --- 函數：解析模型輸出 ---
def extract_answer(text):
    """
    從模型的回答中提取選項數字 (1, 2, 3, 4)。
    DeepSeek R1 會輸出 <think>...</think> 思考過程，需要過濾。
    """
    if not text:
        return None

    # 1. 移除 <think> 標籤內的內容 (DeepSeek R1 特有)
    # flags=re.DOTALL 確保 . 可以匹配換行符號
    text_cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    
    # 2. 策略 A: 尋找明確的 "Answer: X" 格式 (Prompt 中有要求)
    match = re.search(r'Answer:\s*([1-4])', text_cleaned, re.IGNORECASE)
    if match:
        return int(match.group(1))
    
    # 3. 策略 B: 如果沒有標準格式，尋找最後出現的數字 1-4
    # 這是一個備用方案
    numbers = re.findall(r'\b([1-4])\b', text_cleaned)
    if numbers:
        return int(numbers[-1])
        
    return None  # 解析失敗

In [10]:
# --- 主程式 ---
# 1. 讀取資料
print(f"正在讀取 {FILE_PATH}...")
try:
    df = pd.read_excel(FILE_PATH)
except Exception as e:
    print(f"讀取檔案失敗: {e}")

# 檢查必要的欄位
required_columns = ['ID', 'Article', 'Question', 'Option1', 'Option2', 'Option3', 'Option4', 'Answer']
if not all(col in df.columns for col in required_columns):
    print(f"錯誤: Excel 檔案缺少必要的欄位。需要: {required_columns}")

正在讀取 ./data/c3_m_test.xlsx...


In [11]:
df

,ID,Article,Question,Option1,Option2,Option3,Option4,Answer
0,1,一个孕妇在路上行走。一个小女孩走过来问她：“阿姨，您的肚子为什么这么大?”孕妇说：“因为肚子...,请选出与试题内容一致的一项。,小女孩怕麻烦,小女孩认为孕妇怕麻烦,孕妇怕麻烦,孕妇很聪明,2
1,2,很多人年轻时觉得钱比健康重要，老了以后才发现自己错了。很多东西都是这样，等到没有了，我们才明...,我们要重视身体健康。,正确,错误,NaN,NaN,1
2,3,一家公司日前推出一款“懒人座椅”，它视重力为无物，利用了永久磁体，令椅子长时间悬在空中，能给...,“懒人座椅”有什么功能?,健身,减肥,飞翔,美体,1
3,4,一家公司日前推出一款“懒人座椅”，它视重力为无物，利用了永久磁体，令椅子长时间悬在空中，能给...,“懒人座椅”的制作方式是：,软件合成,手工制作,批量生产,电脑操控,2
4,5,一家公司日前推出一款“懒人座椅”，它视重力为无物，利用了永久磁体，令椅子长时间悬在空中，能给...,“懒人座椅”有什么特点?,质量轻巧,价格便宜,存放简单,透明化外观,4
...,...,...,...,...,...,...,...,...
1997,1998,一支探险队进人撒哈拉沙漠，在茫茫沙漠里艰难跋涉。烈日如火，队员们个个又渴又累。这时水喝完了，...,故事想告诉我们什么道理?,沙石可以代替水,要成功应该时刻保存信念,探险队不团结,谎言可以帮人成功,2
1998,1999,近日，美国科学家成功地开发出一项新技术，他们制作了一种特殊的帽子，戴上它就可以使人的大脑与网...,请选出与试题内容一致的一项。,帽子可以代替大脑,大脑可以代替天线,大脑可以成为网络,网络可与大脑连接,4
1999,2000,芳芳今27岁了，她想明年去国外工作。她爸爸同意了，但是妈妈不同意。芳芳的妈妈希望她早点儿结婚。,妈妈不希望芳芳：,找工作,早点儿结婚,结婚太晚,NaN,3
2000,2001,美国的一位数学家近日宣布，他已经发明了两个简单的数学公式，能够用来预测一对新婚夫妇能否维持幸...,请选出与试题内容一致的一项。,公式可以维持婚姻,数学家能预测未来,公式非常准确,公式并不简单,3


In [13]:
# --- 主程式 ---
def main():
    correct_count = 0
    total_questions = len(df)
    results = []

    print(f"開始使用 {MODEL_NAME} 處理 {total_questions} 個問題...")

    # 2. 遍歷每一列進行預測
    for index, row in tqdm(df.iterrows(), total=total_questions):
        article = row['Article']
        question = row['Question']
        # 建立選項字典方便取用
        options = {
            1: row['Option1'],
            2: row['Option2'],
            3: row['Option3'],
            4: row['Option4']
        }
        actual_answer = row['Answer']
        #print(actual_answer)

        # 建構 Prompt
        # 這裡的設計是為了讓 DeepSeek R1 能夠先思考再回答，並最後輸出固定格式
        prompt = f"""
Read the following article and answer the multiple-choice question.

Article:
{article}

Question:
{question}

Options:
1. {options[1]}
2. {options[2]}
3. {options[3]}
4. {options[4]}

Instruction:
Analyze the article and select the best option. 
Finally, output the answer in the exact format: "Answer: X", where X is the option number (1, 2, 3, or 4).
Do not output anything else after the answer.
"""
        # 呼叫模型
        response_text = query_ollama(prompt)
        
        # 解析答案
        predicted_answer = extract_answer(response_text)
        
        # 比對結果
        is_correct = False
        try:
            # 處理 Excel 讀入的答案可能是浮點數或字串的情況
            actual_answer_int = int(actual_answer)
            if predicted_answer is not None and predicted_answer == actual_answer_int:
                correct_count += 1
                is_correct = True
            #else:
                #print("Article:", article)
                #print("Question:", question)
                #print("Option:", "1.", row['Option1'], "2.", row['Option2'], "3.", row['Option3'], "4.", row['Option4'])
                #print("Answer:", actual_answer)
                #print("Predicted Answer:", predicted_answer)
                #print('-'*30)
        except ValueError:
            pass # 忽略無效的標準答案格式

        # 儲存詳細結果
        results.append({
            "ID": row['ID'],
            "Actual": actual_answer,
            "Predicted": predicted_answer,
            "Correct": is_correct,
            "Model_Response": response_text
        })

        #print(predicted_answer)
        if (index+1) % 500 == 0:
            print(correct_count, " / ", (index+1), " = ", 100.0*correct_count/(index+1), "%")

    # 3. 計算並輸出正確率
    accuracy = (correct_count / total_questions) * 100 if total_questions > 0 else 0
    print("\n" + "="*30)
    print(f"處理完成！")
    print(f"總題數: {total_questions}")
    print(f"答對題數: {correct_count}")
    print(f"正確率 (Accuracy): {accuracy:.2f}%")
    print("="*30)

    # 4. 將結果存檔
    output_file = "result_analysis.xlsx"
    pd.DataFrame(results).to_excel(output_file, index=False)
    print(f"詳細結果已儲存至 {output_file}")

if __name__ == "__main__":
    main()

開始使用 gemma3:12b 處理 2002 個問題...


 25%|███████████████████▋                                                           | 500/2002 [05:28<15:29,  1.62it/s]

460  /  500  =  92.0 %


 50%|██████████████████████████████████████▉                                       | 1000/2002 [11:02<14:07,  1.18it/s]

929  /  1000  =  92.9 %


...

1395  /  1500  =  93.0 %


100%|█████████████████████████████████████████████████████████████████████████████▉| 2000/2002 [21:58<00:01,  1.68it/s]

1858  /  2000  =  92.9 %


100%|██████████████████████████████████████████████████████████████████████████████| 2002/2002 [22:00<00:00,  1.52it/s]


處理完成！
總題數: 2002
答對題數: 1860
正確率 (Accuracy): 92.91%
詳細結果已儲存至 result_analysis.xlsx
